# Lab 3.4, Build 1: Attribute every claim

Tina answers a compliance question from case memos and policy text. The harness
splits her answer into claims. You decide, for each claim, which passage it came
from, and you say `UNSUPPORTED` when none of them does.

Run the harness cell, then complete the cell marked **YOUR WORK**.

In [ ]:
# Harness. Nothing here is graded. The grader retrieves, packs, and generates the
# same way, so what you see here is what it sees.
import json
import os
import pathlib
import re
import sys

sys.path.insert(0, "/opt/ara/lib")

from elasticsearch import Elasticsearch
from ara_attrib import split_claims, support

ES = Elasticsearch(os.environ["ES_URL"], api_key=os.environ["ES_API_KEY"],
                   request_timeout=120)
TRACES = pathlib.Path("/home/elastic/.traces")
TRACES.mkdir(exist_ok=True)

COMPLETION_ID = os.environ.get("ARA_INFERENCE_COMPLETION_ID", "cortex-generation")
RERANK_ID = os.environ.get("ARA_RERANK_ID", "")

ANSWER_PROMPT = (
    "You are a compliance assistant for Cortex Bank and Trust. Answer the question "
    "using only the case material and policy text below. Quote every figure exactly "
    "as it appears in the material. If the material does not contain the answer, "
    "reply with the single sentence: I cannot find this information in the case files "
    "or the policy library.\n\n"
    "Material:\n{context}\n\nQuestion: {question}\nAnswer:"
)


def retrieve(query_text, cases=2, policies=1):
    """Case memos and policy chunks for one question, ordered by rerank relevance."""
    passages = []
    plans = (
        ("cortex-cases", "case", {
            "retriever": {"rrf": {"retrievers": [
                {"standard": {"query": {"match": {"body_text": query_text}}}},
                {"standard": {"query": {"semantic": {"field": "body",
                                                     "query": query_text}}}},
            ], "rank_window_size": 50, "rank_constant": 60}},
            "size": cases,
        }),
        ("cortex-policies", "policy", {
            "retriever": {"standard": {"query": {"semantic": {"field": "body",
                                                              "query": query_text}}}},
            "size": policies,
        }),
    )
    for index, source_type, body in plans:
        response = ES.search(index=index, body=body)
        for hit in response["hits"]["hits"]:
            source = hit.get("_source", {})
            text = source.get("body_text") or source.get("body") or ""
            if isinstance(text, dict):
                text = text.get("text", "")
            passages.append({
                "passage_id": str(source.get("case_id") or source.get("doc_id")
                                  or hit["_id"]),
                "source_type": source_type,
                "text": text,
                "score": float(hit.get("_score", 0.0) or 0.0),
            })
    if RERANK_ID and passages:
        response = ES.inference.inference(inference_id=RERANK_ID, body={
            "query": query_text,
            "input": [(p["text"] or "")[:2000] for p in passages],
        })
        body = response.body if hasattr(response, "body") else response
        for entry in body.get("rerank") or []:
            position = int(entry.get("index", -1))
            if 0 <= position < len(passages):
                passages[position]["score"] = float(entry.get("relevance_score", 0.0) or 0.0)
        passages.sort(key=lambda p: p["score"], reverse=True)
    return passages


def pack(passages, per_passage_chars=1400):
    """The context string the answer prompt carries."""
    return "\n\n".join(
        f"[{p['passage_id']} | {p['source_type']}]\n{(p['text'] or '')[:per_passage_chars]}"
        for p in passages
    )


def ask(question, passages):
    """Tina answers at temperature 0 over exactly these passages."""
    response = ES.inference.inference(inference_id=COMPLETION_ID, body={
        "input": ANSWER_PROMPT.format(context=pack(passages), question=question),
        "task_settings": {"temperature": 0},
    })
    body = response.body if hasattr(response, "body") else response
    return (body.get("completion") or [{}])[0].get("result", "") or ""


DEV = json.loads(
    pathlib.Path("/home/elastic/dev-sets/dev-claim-questions.json").read_text()
)
DEV_SAMPLE = 4  # raise this once your attributor is fast enough to be worth it

# A conflict pair to test the two-passages branch without spending a dev question.
CONFLICT_EXAMPLE = {
    "claims": ["A documentary exception may be granted for no more than thirty days "
               "and must be cleared by the relationship manager."],
    "passages": [
        {"passage_id": "case-kyc-gap-018", "source_type": "case", "score": 0.71,
         "text": "The account was opened with a documentary exception for a missing "
                 "corporate resolution. The case note records that a documentary "
                 "exception may be granted for no more than thirty days and must be "
                 "cleared by the relationship manager, and records that the exception "
                 "on this file had stood for ninety-four days."},
        {"passage_id": "policy-004-s5", "source_type": "policy", "score": 0.68,
         "text": "A documentary exception may be granted for no more than thirty days "
                 "and must be cleared by the relationship manager. An exception that "
                 "passes that limit must be reported to the Financial Crimes Compliance "
                 "Unit and the account restricted until the file is complete."},
    ],
}

print(f"{len(DEV)} dev questions available, running the first {DEV_SAMPLE}.")
print(f"completion endpoint: {COMPLETION_ID}")

## What your function is handed

One dev question retrieved, answered, and split, so you can see the shape of
`claims` and `passages` before you write anything.

In [ ]:
# What your function is handed. One dev question, end to end.
question = DEV[0]
passages = retrieve(question["query_text"])

print(f"Q: {question['query_text']}\n")
for p in passages:
    print(f"  {p['passage_id']:<34} {p['source_type']:<7} score {p['score']:.3f}")
    print(f"    {p['text'][:160]}...\n")

answer = ask(question["query_text"], passages)
claims = split_claims(answer)
print(f"Tina's answer ({len(claims)} claims):\n{answer}\n")
for index, claim in enumerate(claims):
    print(f"  claim {index}: {claim}")

print(f"\nGold literal for this question: {question['gold_literal']}")
print("support(claim, passage_text, COMPLETION_ID) returns supports, contradicts, "
      "or neutral.")

## Your attributor

Three outcomes and nothing else: the passage id, `UNSUPPORTED`, or the policy
passage when two passages support the same claim.

In [ ]:
# YOUR WORK: map every claim to the passage that supports it.
import os
import re

from ara_attrib import support

COMPLETION_ID = os.environ.get("ARA_INFERENCE_COMPLETION_ID", "cortex-generation")
FIGURE_RE = re.compile(r"\$?\d[\d,]*(?:\.\d+)?")


def attribute(claims, passages):
    """Return {claim_index: passage_id or "UNSUPPORTED"} for every claim.

    passages entries carry passage_id, source_type ("case" or "policy"), text,
    and score. Every claim needs an entry; a claim you leave out is graded as a miss.
    """
    result = {}
    for index, claim in enumerate(claims):
        # TODO: screen the cheap case first. A figure in the claim that appears in no
        # passage cannot be grounded, and finding that out costs no model call.

        # TODO: ask support() which passages support this claim. Stop early where you
        # can: every call is a model call.

        # TODO: decide what to return. Exactly one supporter is easy. No supporter has
        # one honest value. More than one supporter is the conflict case, and
        # source_type is how you tell the authority from the evidence.
        result[index] = "UNSUPPORTED"
    return result

## Run the dev sample

Then read the claims your function marked `UNSUPPORTED` against the passages it
was given. That list is the whole of the feedback loop here.

In [ ]:
# Run the dev sample, then read what came back UNSUPPORTED.
CLAIMS_TOTAL = TO_CASE = TO_POLICY = UNSUPPORTED = 0
UNSUPPORTED_EXAMPLES = []

for question in DEV[:DEV_SAMPLE]:
    passages = retrieve(question["query_text"])
    by_id = {p["passage_id"]: p for p in passages}
    answer = ask(question["query_text"], passages)
    claims = split_claims(answer)
    mapping = attribute(claims, passages)

    print(f"\n{question['query_id']}  {len(claims)} claims  "
          f"gold {question['gold_literal']}")
    for index, claim in enumerate(claims):
        passage_id = mapping.get(index, "UNSUPPORTED")
        CLAIMS_TOTAL += 1
        if passage_id == "UNSUPPORTED":
            UNSUPPORTED += 1
            UNSUPPORTED_EXAMPLES.append(claim)
            marker = "UNSUPPORTED"
        elif by_id.get(passage_id, {}).get("source_type") == "policy":
            TO_POLICY += 1
            marker = f"policy  {passage_id}"
        else:
            TO_CASE += 1
            marker = f"case    {passage_id}"
        print(f"  claim {index}  {marker}")
        print(f"            {claim[:110]}")

print(f"\n{CLAIMS_TOTAL} claims: {TO_CASE} to a memo, {TO_POLICY} to a policy "
      f"passage, {UNSUPPORTED} unsupported.")
print("\nCheck each unsupported claim against the passages printed above. A claim you "
      "can see in a passage means a supports verdict was thrown away.")

# The conflict branch, on a fixed pair so you can test it without a dev question.
conflict = attribute(CONFLICT_EXAMPLE["claims"], CONFLICT_EXAMPLE["passages"])
print(f"\nConflict example returned: {conflict.get(0)}")
print("Both passages support that claim. One of them is the authority.")

In [ ]:
# Save the results file the grader reads.
payload = {
    "dev_questions": DEV_SAMPLE,
    "claims_total": CLAIMS_TOTAL,
    "attributed_to_case": TO_CASE,
    "attributed_to_policy": TO_POLICY,
    "unsupported": UNSUPPORTED,
    "unsupported_examples": UNSUPPORTED_EXAMPLES[:5],
}
(TRACES / "attribution-results.json").write_text(json.dumps(payload, indent=2))
print(f"attribution-results.json written with {CLAIMS_TOTAL} claims.")
print("Select Check.")